# Basic Usage

Also good to check out Stim's getting started notebook: https://github.com/quantumlib/Stim/blob/main/doc/getting_started.ipynb

## Imports

In [5]:
# from pathlib import Path
# import sys
# import IPython

# # Get the current working directory (where the notebook is running)
# notebook_dir = Path(IPython.get_ipython().run_line_magic('pwd', '')).resolve()

# # Add the project root to sys.path
# sys.path.append(str(notebook_dir.parent))

import numpy as np
import pymatching
import stim 
import sinter

from corrqec2.experiments import Experiment, SurfaceCodeMemory, error_matrix_shape, get_noisy_qubits, format_noisy_qubits
from corrqec2.noisemodels import NoiseModel, StandardCircuitLevel
from corrqec2 import Sampler

# from src.experiments import Experiment, SurfaceCodeMemory, error_matrix_shape, get_noisy_qubits, format_noisy_qubits
# from src.noisemodels import NoiseModel, StandardCircuitLevel
# from src import Sampler

ENVISAGED USAGE

In [ ]:
# Choose QEC experiment
experiment = SurfaceCodeMemory(distance=3, rounds='3d', memory_type='Z')
# Choose noise model
noise_model = Streaky(model_params={'A': 1, 'q': 0.01, 'r': 2})

# Monte-carlo sampling
sampler = Sampler(experiment, noise_model)
detection_events, observable_flips = sampler.sample(num_samples=1000)

In [8]:
error_matrix= None

In [ ]:
experiment = SurfaceCodeMemory(distance=3, rounds='3d', memory_type='Z')
detection_events, observable_flips = sampler.sample_with_custom_errors(experiment, error_matrix)

## Example: Directly sample from an input error matrix

In [6]:
# First generate experiment
distance = 3
rounds = 'd'  # Number of rounds, can be an integer or a string '{n}d' to represent n times d rounds
memory_type = 'Z'  # Z or X type surface code memory experiment
experiment = SurfaceCodeMemory(distance=distance, rounds=rounds, memory_type=memory_type)
# This is just a noiseless stim circuit
circuit = experiment.gen_stim_circuit()

Can specify to include gate-level independent errors which are sampled by Stim

In [7]:
noisy_qubit_types = ['data', 'syndrome'] # Can specify 'all', 'data', 'syndrome', or a list of strings

# For now don't include gate noise
# p = 0.001
# gate_noise = {
#             'after_identity_depolarization': p,
#             'after_clifford_depolarization': p,
#             'before_measure_flip_probability': p,
#             'after_reset_flip_probability': p,
#         }
gate_noise = None

You can print the error matrix dimensions for the experiment and the ordering of which index in the error matrix corresponds to which qubit

In [8]:
# (Input) error matrix dimensions
error_matrix_dims = error_matrix_shape(experiment, noisy_qubit_types)
print(f"Error matrix dimensions: {error_matrix_dims}")

# You can print out the ordering of which index in the error matrix corresponds to which qubit
print(f"Error matrix mapping: {format_noisy_qubits(experiment, noisy_qubit_types)}")

Error matrix dimensions: (17, 3)
Error matrix mapping: ['data(1.0, 1.0)', 'syndrome(2.0, 0.0)', 'data(3.0, 1.0)', 'data(5.0, 1.0)', 'data(1.0, 3.0)', 'syndrome(2.0, 2.0)', 'data(3.0, 3.0)', 'syndrome(4.0, 2.0)', 'data(5.0, 3.0)', 'syndrome(6.0, 2.0)', 'syndrome(0.0, 4.0)', 'data(1.0, 5.0)', 'syndrome(2.0, 4.0)', 'data(3.0, 5.0)', 'syndrome(4.0, 4.0)', 'data(5.0, 5.0)', 'syndrome(4.0, 6.0)']


error_matrix can either be a 3D array (n_samples, n_noisy_qubits, n_rounds) or a 2D array (n_noisy_qubits, n_rounds)

In [9]:
n_samples = 20  # Number of samples
dims = (n_samples, *error_matrix_dims)

Let's start with independent Pauli flips

In [10]:
p = 0.05  # Probability of random Pauli flip
random_error = np.random.random(dims) < p
random_pauli = np.random.randint(1, 4, dims)
error_matrix = random_error * random_pauli
# print(error_matrix)

Detection events contains the error syndrome (to be input into your decoder), and observable_flips is a boolean on whether the logical observable was flipped or not. The job of the decoder is to predict whether the observable was flipped or not based on the error syndrome. A logical error occurs if prediction != observable flip.

In [11]:
# We will construct our matching graph for MWPM based on a standard circuit level noise model
# More on this later
scl = StandardCircuitLevel(p = p)  # StandardCircuitLevel just sets the same p for all gate_noise keys
scl_circuit = scl.gen_noisy_circuit(experiment)

# Can view the scl_circuit (its just a stim circuit)
# scl_circuit.diagram('timeline-svg')

# Get the detector error model
scl_detector_error_model = scl_circuit.detector_error_model()

# Configure MWPM decoder based on the detector error model (will be used to specify the matching graph)
matcher = pymatching.Matching.from_detector_error_model(scl_detector_error_model)

In [12]:
# Sample detection events and observable flips using stim.FlipSimulator
detection_events, observable_flips = Sampler.sample_with_custom_errors(experiment=experiment, 
                                                                       error_matrix=error_matrix, 
                                                                       gate_noise=gate_noise, 
                                                                       noisy_qubit_types=noisy_qubit_types)

# Get decoder predictions
predictions = matcher.decode_batch(detection_events).flatten().astype(bool)

# Let's see how succesful we were
def print_predictions_vs_truth(predictions, observable_flips):
    print("Sample: predicted, actual, error")
    for i in range(predictions.shape[0]):
        print(f"{i}: {predictions[i]}, {observable_flips[i]}, {predictions[i] != observable_flips[i]}")
    print("=" * 20)
    print(f"Number of samples: {predictions.shape[0]}")
    print(f"Number of errors: {np.sum(predictions != observable_flips)}")
    # print(f"Logical error rate: {calc_logical_error_rate(observable_flips, predictions)}")

def calc_logical_error_rate(observable_flips, predictions):
    return np.mean(observable_flips != predictions)

print_predictions_vs_truth(predictions, observable_flips)

Sample: predicted, actual, error
0: True, True, False
1: False, False, False
2: True, True, False
3: False, False, False
4: False, False, False
5: False, False, False
6: False, False, False
7: True, True, False
8: False, False, False
9: False, False, False
10: False, False, False
11: False, False, False
12: False, False, False
13: False, False, False
14: False, False, False
15: False, False, False
16: False, False, False
17: False, False, False
18: True, True, False
19: False, False, False
Number of samples: 20
Number of errors: 0


# Test with streaky error model

Instead of directly sampling from an input error matrix using `Sampler.sample_with_custom_errors`, you can define a subclass `CustomNoiseModel` that inherits from the base `NoiseModel` class. `Sampler` will be able to access this `NoiseModel` to automatically run sampling experiments (WIP). You will need to write a `gen_error_matrix(self)` method for the subclass, and if you want to compare with a marginalised independent model, you will want to write `gen_marginalised_circuit(self)`.

In [13]:
class CustomNoiseModel(NoiseModel):
    def __init__(self, gate_noise: dict | None = None, noisy_qubit_types: str | list[str] = 'all'):
        super().__init__(gate_noise=gate_noise, noisy_qubit_types=noisy_qubit_types)
        self._no_error_matrix = False

    def gen_error_matrix(self, experiment: Experiment, n_samples: int = 1) -> np.ndarray:
        """
        Generate the error matrix for a given experiment and number of samples. The dimension of the error 
        matrix can be accessed using the function `error_matrix_shape(experiment, self.noisy_qubit_types)`. 
        The error matrix is a numpy array of shape `(n_samples, n_qubits, n_timesteps)`.
        """
        # YOUR CODE HERE
        raise NotImplementedError("This method should be implemented in a subclass.")
    
    def gen_marginalised_circuit(self, experiment: Experiment) -> stim.Circuit:
        """Generate the marginalised circuit for a given experiment. The output will be a Stim circuit object."""
        # YOUR CODE HERE
        raise NotImplementedError("This method should be implemented in a subclass.")

Example with streaky model where during a streak, a qubit experiences a bit-flip (X error) every round, and the length of the streak decays polynomially.

In [14]:
class Streaky(NoiseModel):
    def __init__(self, model_params: dict,  gate_noise: dict | None = None, noisy_qubit_types: str | list[str] = 'all'):
        super().__init__(gate_noise=gate_noise, noisy_qubit_types=noisy_qubit_types)
        self.model_params = model_params
        self._no_error_matrix = False

    def gen_error_matrix(self, experiment: Experiment, n_samples: int = 1) -> np.ndarray:
        A = self.model_params['A']
        q = self.model_params['q']
        n = self.model_params['n']
        n_timesteps = experiment.rounds
        n_qubits = len(get_noisy_qubits(experiment, self.noisy_qubit_types))
        return sample_streaky_errors(A=A, q=q, n=n, num_timesteps=n_timesteps, num_samples=n_samples * n_qubits).reshape((n_samples, n_qubits, n_timesteps))

    def gen_marginalised_circuit(self, experiment: Experiment) -> stim.Circuit:
        # WIP
        # For now just return a circuit with standard circuit level noise
        circuit = StandardCircuitLevel(p=self.model_params['q']).gen_noisy_circuit(experiment)
        pass
        
    def _calc_marginals(self, experiment: Experiment) -> dict:
        pass

# Borrowing sample_streaky_errors
def sample_streaky_errors(A, q, n, num_timesteps, num_samples=1):
    """SOLID STREAK OF X ERRORS WITH POLYNOMIALLY DECAYING CORRELATIONS"""
    rng = np.random.default_rng()
    
    # Calculate pair probabilities
    t_list = list(range(num_timesteps))
    t_pair_distances = np.array([b - a for i, b in enumerate(t_list) for a in t_list[:i]])
    t_pair_probs = A * q / (t_pair_distances ** n)
    
    t_pair_probs_array = np.tile(t_pair_probs, (num_samples, 1))
    
    # Sample pair errors
    rnd = rng.random(t_pair_probs_array.shape)
    t_pair_errors = (rnd < t_pair_probs_array).astype(bool)
    
    # Convert pair errors to single time errors
    map = gen_pair_to_time_map(num_timesteps)
    time_errors = np.matmul(t_pair_errors, map)
    
    # Convert mixing errors to bit flip errors (50% chance of a bit flip if the time error is present)
    # random_bits = rng.integers(0, 1, size=time_errors.shape, endpoint=True)
    # error_matrix = np.logical_and(time_errors, random_bits)
    
    return time_errors

def get_t_pairs(num_timesteps):
    round_list = list(range(num_timesteps))
    return np.array([(a, b) for i, b in enumerate(round_list) for a in round_list[:i]])

def gen_pair_to_time_map(num_timesteps):
    t_pairs = get_t_pairs(num_timesteps)
    pair_to_time_map = np.zeros((len(t_pairs), num_timesteps), dtype=bool)
    
    for i, (a, b) in enumerate(t_pairs):
        pair_to_time_map[i, a:b+1] = 1
        
    return pair_to_time_map

In [15]:
distance = 3
rounds = '3d'
memory_type = 'Z'  # Z or X type surface code memory experiment

experiment = SurfaceCodeMemory(distance=distance, rounds=rounds, memory_type=memory_type)
experiment.gen_stim_circuit()

model_params = {'A': 1, 
                'q': 0.1,
                'n': 2}

# For now don't include gate noise
# p = 0.001
# gate_noise = {
#             'after_identity_depolarization': p,
#             'after_clifford_depolarization': p,
#             'before_measure_flip_probability': p,
#             'after_reset_flip_probability': p,
#         }
gate_noise = None
noisy_qubit_types = ['data', 'syndrome']

streaky = Streaky(model_params=model_params, gate_noise=gate_noise, noisy_qubit_types=noisy_qubit_types)

In [24]:
sampler = Sampler(experiment=experiment,
                  noise_model=streaky,
                  batch_size=1000,  # Adjust the number of simultaneous simulations
                  )

WIP

In [25]:
detection_events, observable_flips = sampler._sample_batch()